**Note:** Update the file paths to your own data location before running.

In [ ]:
from google.colab import drive

import pandas as pd
import operator
import numpy as np
from sklearn.metrics import r2_score
import random
import copy
import math
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

In [ ]:
drive.mount('/content/drive')

file_path = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_1obj_r2 of GP/train219_dataset.csv"
df = pd.read_csv(file_path)


FEATURE_COLUMNS = ['Soil', 'Slope', 'Rain', 'LULC', 'Lithology','Lineament density', 'Geomorphology', 'Drainage density']
# #FEATURE_COLUMNS = ['geomorphology','Rain','drainage density']
TARGET_COLUMN = 'Net Groundwater Recharge'

X = df[FEATURE_COLUMNS].values
y = df[TARGET_COLUMN].values
validation_labels = df['validation point categorical data'].values



# Filter dataframe to use selected features
#df_filtered = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()

# Optional: filter out extreme target values (e.g., outliers above 300)
#df_filtered = df_filtered[df_filtered[TARGET_COLUMN] < 300]

# Extract X and y
#X = df_filtered[FEATURE_COLUMNS].values
#y = df_filtered[TARGET_COLUMN].values

#Inspect shapes and first few rows
#print("Shape of X:", X.shape)
#print("Shape of y:", y.shape)
#print("Sample features (X):", X[:3])
#print("Sample target values (y):", y[:3])

# Mapping: x0 -> 'Soil', x1 -> 'Slope', etc.
symbolic_variable_map = {f'x{i}': FEATURE_COLUMNS[i] for i in range(len(FEATURE_COLUMNS))}

print("Symbolic Variable Mapping:")
for symbol, name in symbolic_variable_map.items():
     print(f"{symbol} → {name}")

In [ ]:
def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def safe_exp(x):
    # Prevent overflow
    return math.exp(clamp(x, -20, 20))

def safe_pow(x, y):
    # Prevent extreme exponents and huge bases
    base = abs(x) + 1e-6
    expn = clamp(y, -3, 3)
    return math.pow(base, expn)

# Protected division to avoid divide-by-zero
def protected_div(x, y):
    try:
        return x / y if abs(y) > 1e-6 else 1.0
    except:
        return 1.0

OPERATORS = {
    '+': (operator.add, 2),
    '-': (operator.sub, 2),
    '*': (operator.mul, 2),
    '/': (protected_div, 2),
    'neg': (lambda x: -x, 1),
    'sin': (math.sin, 1),
    'cos': (math.cos, 1),
    #'exp': (math.exp, 1),
    'log': (lambda x: math.log(abs(x) + 1e-6), 1),  # protected log
    'square': (lambda x: x * x, 1),
    'cube': (lambda x: x * x * x, 1),
    'exp': (safe_exp, 1),
    'pow': (safe_pow, 2),
}

class Node:
    def __init__(self, name, children=None):
        self.name = name
        self.children = children if children else []

    def copy(self):
        return Node(self.name, [child.copy() for child in self.children]) if self.children else Node(self.name)


    def evaluate(self, X_row, feature_map=None):
        if self.name in OPERATORS:
            func, arity = OPERATORS[self.name]
            if len(self.children) != arity:
                raise ValueError(f"Invalid expression: Operator '{self.name}' expects {arity} children, got {len(self.children)}")
            args = [child.evaluate(X_row, feature_map) for child in self.children]
            return func(*args)
        elif feature_map and self.name in feature_map:
            return X_row[feature_map[self.name]]
        elif isinstance(self.name, str) and self.name.startswith('x'):
            idx = int(self.name[1:])
            return X_row[idx]
        else:
            return float(self.name)

    # def count_nodes(self):
    #     return 1 + sum(child.count_nodes() for child in self.children)

    def to_string(self, feature_names=None):
        if self.name in OPERATORS:
            return f"{self.name}({', '.join(child.to_string(feature_names) for child in self.children)})"
        elif isinstance(self.name, str) and self.name.startswith('x') and feature_names:
            idx = int(self.name[1:])
            return feature_names[idx]
        else:
            return str(self.name)

    def __str__(self):
        return self.to_string()

    def weighted_complexity(self, parent_is_nonlinear=False, depth=1):
        LINEAR_OPS = {'+', '-', 'neg'}
        MILD_NONLINEAR_OPS = {'*'}
        STRONG_NONLINEAR_OPS = {'/', 'log', 'exp', 'pow', 'sin', 'cos', 'square', 'cube'}

#feature=0, +,=,neg=1, *=1.25, /,sin,cos,log,exp,square,cube=1.5, pow=2, constant=0.5
        if isinstance(self.name, str) and self.name.startswith('x'):
            cost = 0
            is_nonlinear = False
        elif self.name in LINEAR_OPS:
            cost = 1
            is_nonlinear = False
        elif self.name in MILD_NONLINEAR_OPS:
            cost = 1.25
            is_nonlinear = False
        elif self.name in STRONG_NONLINEAR_OPS:
            if self.name == '/':
                cost = 1.5
            elif self.name in ['square', 'cube']:
                cost = 1.5 if self.name == 'square' else 2
            elif self.name in ['sin', 'cos', 'log', 'exp']:
                cost = 1.25
            else:  # pow
                cost = 2
            is_nonlinear = True
        elif is_constant(self.name):
            cost = 0.5
            is_nonlinear = False
        else:
            cost = 1
            is_nonlinear = False

        # 2. Penalize ONLY problematic cases:
        # Very deep trees (>15 levels)
        depth_penalty = 0.1 * max(0, depth - 15)  # Applies to ALL nodes

        # Redundant nesting (sin(cos)), but not single nonlinear terms
        recursive_penalty = 0.05 if (parent_is_nonlinear and self.name in STRONG_NONLINEAR_OPS) else 0

        total = cost + depth_penalty + recursive_penalty
        for child in self.children:
            total += child.weighted_complexity(self.name in STRONG_NONLINEAR_OPS, depth + 1)
        return total

def is_constant(name):
    #Check if node is a numeric constant (handles negatives and floats)
    if isinstance(name, (int, float)):
        return True
    try:
        float(name)
        return True
    except (ValueError, TypeError):
        return False




def to_readable_string(self, var_map):
    if self.name in OPERATORS:
        return f"{self.name}({', '.join(child.to_readable_string(var_map) for child in self.children)})"
    else:
        return var_map.get(self.name, str(self.name))


# Attach to Node class
Node.to_readable_string = to_readable_string



In [ ]:
def generate_random_tree(max_depth, feature_count, current_depth=0, parent_op=None):
    # Make deeper trees more likely
    terminal_prob = 0.1 + 0.4 * (current_depth / max_depth) ## (more terminals → smaller trees)

    if current_depth >= max_depth or random.random() < terminal_prob:
        # Prefer features (70%) over constants (30%)
        if random.random() < 0.7:
            return Node(f'x{random.randint(0, feature_count - 1)}')
        else:
            return Node(str(round(random.uniform(-5, 5), 2)))
    else:
        # Add operator variety
        op_weights = {
            '+': 0.18, '-': 0.18, '*': 0.20, '/': 0.07,
            'sin': 0.06, 'cos': 0.06,
            'log': 0.06, 'exp': 0.04,
            'square': 0.06, 'cube': 0.03,
            'pow': 0.06
        }



        op = random.choices(list(op_weights.keys()), weights=list(op_weights.values()), k=1)[0]
        func, arity = OPERATORS[op]
        children = [generate_random_tree(max_depth, feature_count, current_depth + 1, op) for _ in range(arity)]
        return Node(op, children)


In [ ]:
# Recursively collect all nodes along with their parent nodes
def collect_nodes_with_parents(node, parent=None):
    nodes = [(node, parent)]
    for child in node.children:
        nodes.extend(collect_nodes_with_parents(child, node))
    return nodes

## Crossover
def crossover(tree1, tree2):
    child1 = copy.deepcopy(tree1)
    child2 = copy.deepcopy(tree2)

    # Collect all nodes with their parent references
    nodes1 = collect_nodes_with_parents(child1)
    nodes2 = collect_nodes_with_parents(child2)

    # Prefer internal nodes (i.e., operators), fallback to any node
    op_nodes1 = [n for n in nodes1 if n[0].name in OPERATORS]
    op_nodes2 = [n for n in nodes2 if n[0].name in OPERATORS]

    if op_nodes1 and op_nodes2:
        (target1, parent1), (target2, parent2) = random.choice(op_nodes1), random.choice(op_nodes2)
    else:
        (target1, parent1), (target2, parent2) = random.choice(nodes1), random.choice(nodes2)

    # Swap the subtrees
    if parent1 is None:
        child1 = target2
    else:
        idx = parent1.children.index(target1)
        parent1.children[idx] = target2

    if parent2 is None:
        child2 = target1
    else:
        idx = parent2.children.index(target2)
        parent2.children[idx] = target1

    # print crossover to check
    # print(f"[CROSSOVER] Parent1: {tree1.to_string()}")
    # print(f"[CROSSOVER] Parent2: {tree2.to_string()}")
    # print(f"[CROSSOVER] Child1 : {child1.to_string()}")
    # print(f"[CROSSOVER] Child2 : {child2.to_string()}")

    return child1, child2


# # # Mutation
def mutate(tree, max_depth, feature_count):
    #tree_copy = copy.deepcopy(tree)
    tree_copy = tree.copy()  # assuming you have a custom .copy() method
    nodes = collect_nodes_with_parents(tree_copy)
    target_node, parent = random.choice(nodes)

    mutation_type = random.choice(['subtree', 'operator', 'constant', 'feature'])

    if mutation_type == 'subtree':
        new_subtree = generate_random_tree(max_depth=max_depth, feature_count=feature_count)
        if parent is None:
            return new_subtree
        else:
            idx = parent.children.index(target_node)
            parent.children[idx] = new_subtree

    elif mutation_type == 'operator' and target_node.name in OPERATORS:
        current_op = target_node.name
        valid_ops = [op for op in OPERATORS if OPERATORS[op][1] == OPERATORS[current_op][1] and op != current_op]
        if valid_ops:
            new_op = random.choice(valid_ops)
            target_node.name = new_op

    elif mutation_type == 'constant' and is_constant(target_node.name):
        new_val = round(random.uniform(-10, 10), 2)
        target_node.name = new_val

    elif mutation_type == 'feature' and isinstance(target_node.name, str) and target_node.name.startswith('x'):
        old_feature = target_node.name
        new_feature = old_feature
        while new_feature == old_feature:
            new_feature = f'x{random.randint(0, feature_count - 1)}'
        target_node.name = new_feature
        #print(f"[MUTATION] Replaced feature {old_feature} → {new_feature}")

    return tree_copy



In [ ]:
class Individual:
    def __init__(self, tree, X, y, feature_names, validation_labels=None):
        self.tree = tree
        self.X = X
        self.y = y
        self.feature_names = feature_names
        self.validation_labels = validation_labels

        self.rank = None
        self.crowding_distance = 0.0

        self._evaluate()
        #print(f"[Debug] Feature names: {self.feature_names}")


    def _evaluate(self):
        try:
            # Evaluate predictions
            y_pred = np.clip([self.tree.evaluate(row) for row in self.X], -1e3, 1e3)
            self.r2 = r2_score(self.y, y_pred)
            self.predicted = y_pred
        except Exception:
            self.r2 = -np.inf
            self.predicted = [None] * len(self.y)

        # complexity calculation
        #self.complexity = self.tree.count_nodes()
        self.complexity = self.tree.weighted_complexity()



    @property
    def fitness(self):
        return -self.r2  # minimize negative R² (maximize R²)

    @property
    def symbolic(self):
        return str(self.tree)

    @property
    def readable(self):
        return self.tree.to_readable_string(self.feature_names)


    def copy(self):
        return Individual(copy.deepcopy(self.tree), self.X, self.y, self.feature_names, self.validation_labels)

    # def score(self, alpha=1.0, beta=0.01):
    #     return alpha * self.r2 - beta * self.complexity


In [ ]:
# NSGA-II survivor selection using non-dominated sorting and crowding distance

def objective_value(ind, obj):
    if obj == 'r2':
        return -ind.r2   # convert R² to minimization
    elif obj == 'complexity':
        return ind.complexity
    else:
        return getattr(ind, obj)


def compute_crowding_distance(front, objectives):
    size = len(front)
    if size == 0:
        return {}

    distance = {ind: 0.0 for ind in front}

    for obj in objectives:
        sorted_front = sorted(front, key=lambda ind: objective_value(ind, obj))

        distance[sorted_front[0]] = float('inf')
        distance[sorted_front[-1]] = float('inf')

        obj_min = objective_value(sorted_front[0], obj)
        obj_max = objective_value(sorted_front[-1], obj)

        if obj_max == obj_min:
            continue

        for j in range(1, size - 1):
            prev_val = objective_value(sorted_front[j - 1], obj)
            next_val = objective_value(sorted_front[j + 1], obj)

            distance[sorted_front[j]] += (next_val - prev_val) / (obj_max - obj_min)

    return distance



def select_survivors(population, num_survivors):
    # Sort population by R² descending
    return sorted(population, key=lambda ind: ind.r2, reverse=True)[:num_survivors]


def tournament_selection_r2(population, k, tournament_size=2):
    ## Single-objective tournament selection based only on R².

    selected = []
    for _ in range(k):
        contenders = random.sample(population, tournament_size)
        winner = max(contenders, key=lambda ind: ind.r2)
        selected.append(winner)
    return selected



In [ ]:
def generate_offspring(parents, num_offspring, max_depth, feature_count, mutation_rate=0.3):
    offspring = []
    while len(offspring) < num_offspring:
        p1, p2 = random.sample(parents, 2)

        # Crossover
        child1_tree, child2_tree = crossover(p1.tree, p2.tree)
        print(f"[CROSSOVER] Parent1: {p1.tree.to_string()}")
        print(f"[CROSSOVER] Parent2: {p2.tree.to_string()}")
        print(f"[CROSSOVER] Child1 : {child1_tree.to_string()}")
        print(f"[CROSSOVER] Child2 : {child2_tree.to_string()}")

        child1 = Individual(child1_tree, p1.X, p1.y, p1.feature_names)
        child2 = Individual(child2_tree, p2.X, p2.y, p2.feature_names)

        # Mutation for Child1
        if random.random() < mutation_rate:
            if random.random() < 0.05: # only 5% chance of full reset
                new_tree = generate_random_tree(max_depth, feature_count)
                print(f"[MUTATION] Child1 replaced by new random tree: {new_tree.to_string()}")
                child1 = Individual(new_tree, p1.X, p1.y, p1.feature_names)
            else:
                print(f"[MUTATION] Before mutation (Child1): {child1.tree.to_string()}")
                mutated_tree = mutate(child1.tree, max_depth, feature_count)
                print(f"[MUTATION] After mutation  (Child1): {mutated_tree.to_string()}")
                child1 = Individual(mutated_tree, p1.X, p1.y, p1.feature_names)

        # Mutation for Child2
        if random.random() < mutation_rate:
            if random.random() < 0.05:  # only 5% chance of full reset
                new_tree = generate_random_tree(max_depth, feature_count)
                print(f"[MUTATION] Child2 replaced by new random tree: {new_tree.to_string()}")
                child2 = Individual(new_tree, p2.X, p2.y, p2.feature_names)
            else:
                print(f"[MUTATION] Before mutation (Child2): {child2.tree.to_string()}")
                mutated_tree = mutate(child2.tree, max_depth, feature_count)
                print(f"[MUTATION] After mutation  (Child2): {mutated_tree.to_string()}")
                child2 = Individual(mutated_tree, p2.X, p2.y, p2.feature_names)

        offspring.append(child1)
        if len(offspring) < num_offspring:
            offspring.append(child2)

    return offspring[:num_offspring]


In [ ]:
## pareto front method


###TEST #####.   feature track per generation
from collections import Counter

feature_usage_log = []  # Global tracker


def count_features(tree):
    if isinstance(tree.name, str) and tree.name.startswith('x'):
        return Counter([tree.name])
    counts = Counter()
    for child in tree.children:
        counts += count_features(child)
    return counts


In [ ]:
all_logs = []


In [ ]:
def create_individual_population(n_individuals, max_depth, feature_count, X, y, feature_names):
    population = []
    for _ in range(n_individuals):
        tree = generate_random_tree(max_depth=max_depth, feature_count=feature_count)
        ind = Individual(tree, X, y, feature_names)
        population.append(ind)
    return population

#all_logs = []

def run_gp_evolution(
    X, y, feature_names,
    population_size=100,
    generations=100,
    max_depth=6,
    mutation_rate=.3
):
    # Initial population
    population = create_individual_population(
        n_individuals=population_size,
        max_depth=max_depth,
        feature_count=len(feature_names),
        X=X,
        y=y,
        feature_names=feature_names
    )

    gen_expr_log_path = '/content/drive/My Drive/GWL/r2_complexity/train219_val99_1obj_r2 of GP/train219_all_generation_expressions_log.txt'
    gen_expr_log = open(gen_expr_log_path, 'w')

    best_overall = None

    for gen in range(generations):

        parents = tournament_selection_r2(
            population,
            k=population_size,
            tournament_size=2
        )

        offspring = generate_offspring(
            parents=parents,
            num_offspring=population_size,
            max_depth=max_depth,
            feature_count=len(feature_names),
            mutation_rate=mutation_rate
        )

        combined_population = population + offspring

        population = sorted(
            combined_population,
            key=lambda ind: ind.r2,
            reverse=True
        )[:population_size]

        gen_best = max(population, key=lambda ind: ind.r2)

        if best_overall is None or gen_best.r2 > best_overall.r2:
            best_overall = gen_best

        gen_header = f"\nGeneration {gen + 1}"
        print(gen_header)
        gen_expr_log.write(gen_header + '\n')

        for i, ind in enumerate(population):
            expr_line = f"Individual {i + 1}, R²: {ind.r2:.4f}, Complexity: {ind.complexity:.4f}, Expr: {ind.readable}"
            print(expr_line)
            gen_expr_log.write(expr_line + '\n')

        generation_usage = Counter()
        for ind in population:
            generation_usage += count_features(ind.tree)
        feature_usage_log.append(generation_usage)

        for i, ind in enumerate(population):
            try:
                y_pred = [ind.tree.evaluate(row) for row in ind.X]
            except Exception:
                y_pred = [None] * len(ind.y)

            target_str = ",".join([f"{v:.4f}" for v in ind.y])
            pred_str = ",".join([f"{v:.4f}" if v is not None else "NaN" for v in y_pred])

            all_logs.append({
                "Generation": gen + 1,
                "Individual": i,
                "R2": ind.r2,
                "Complexity": ind.complexity,
                "Targets": target_str,
                "Predictions": pred_str,
            })

    gen_expr_log.close()
    return population, best_overall

In [ ]:

#########

# =========================
# MULTI-RUN WITH BEST EXPRESSION FROM ALL -20 RUNS
# =========================

import os, random
import numpy as np

import sys
sys.setrecursionlimit(200000)

NUM_RUNS = 20  ##20
BASE_SEED = 42
SEEDS = [BASE_SEED + i for i in range(NUM_RUNS)]

OUT_DIR = "/content/drive/My Drive/GWL/r2_complexity/train219_val99_1obj_r2 of GP"
os.makedirs(OUT_DIR, exist_ok=True)

ALL_RUNS_BEST_TXT = os.path.join(OUT_DIR, "single_obj_r2_best_per_run.txt")

# clear file once
with open(ALL_RUNS_BEST_TXT, "w") as f:
    f.write("SINGLE-OBJECTIVE GP (maximize R²)\n")
    f.write("="*70 + "\n\n")

best_per_run = []   # store best_overall from each run

for run_idx, seed in enumerate(SEEDS, start=1):
    print(f"\n\n**RUN {run_idx}/{NUM_RUNS}")


    random.seed(seed)
    np.random.seed(seed)

    # Run one GP evolution
    # Run one GP evolution (SAFE)
    try:
        final_pop, best_overall = run_gp_evolution(
            X=X,
            y=y,
            feature_names=symbolic_variable_map,
            population_size=100,
            generations=100,
            max_depth=6,
            mutation_rate=0.3
        )
    except RecursionError as e:
        msg = f"Run {run_idx:02d} | RecursionError inside GP | seed={seed} | {e}"
        print(msg)
        with open(ALL_RUNS_BEST_TXT, "a") as f:
            f.write(msg + "\n")
        continue

    best_per_run.append(best_overall)

    # Safe readable (readable can also trigger recursion)
    try:
        readable_expr = best_overall.readable
    except RecursionError:
        readable_expr = "<RecursionError while building readable expression>"

    line = (
        f"\nRun {run_idx:02d} | "
        f"Best R2: {best_overall.r2:.6f} | "
        f"Complexity: {best_overall.complexity:.6f} | "
        f"Expr: {readable_expr}"
    )
    print(line)

    # ---- append to single txt ----
    with open(ALL_RUNS_BEST_TXT, "a") as f:
        f.write(line + "\n")


    # ======================================================
    # SAVE BEST RESULT FOUND SO FAR AFTER EVERY COMPLETED RUN
    # ======================================================
    BEST_SO_FAR_TXT = os.path.join(
        OUT_DIR,
        "single_obj_best_so_far.txt"
    )

    best_so_far = max(best_per_run, key=lambda ind: ind.r2)

    try:
        best_so_far_expr = best_so_far.readable
    except RecursionError:
        best_so_far_expr = "<RecursionError while building readable expression>"

    with open(BEST_SO_FAR_TXT, "w", encoding="utf-8") as f:
        f.write("BEST SINGLE-OBJECTIVE RESULT FOUND SO FAR\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Successful runs: {len(best_per_run)}\n")
        f.write(f"Latest completed run: {run_idx}\n")
        f.write(f"R2: {best_so_far.r2:.6f}\n")
        f.write(f"Complexity: {best_so_far.complexity:.6f}\n")
        f.write(f"Expression: {best_so_far_expr}\n")


# ---- best across all runs ----
print("\n\n **BEST ACROSS ALL RUNS**")

if not best_per_run:
    print("No successful runs (all failed due to RecursionError).")
else:
    best_across_runs = max(best_per_run, key=lambda ind: ind.r2)

    # Safe readable (important!)
    try:
        readable_expr = best_across_runs.readable
    except RecursionError:
        readable_expr = "<RecursionError while building readable expression>"

    print(
        f"Best R2: {best_across_runs.r2:.6f} | "
        f"Complexity: {best_across_runs.complexity:.6f} | "
        f"Expr: {readable_expr}"
    )

    with open(ALL_RUNS_BEST_TXT, "a") as f:
        f.write("\n\n**BEST ACROSS ALL RUNS**\n")
        f.write(
            f"Best R2: {best_across_runs.r2:.6f} | "
            f"Complexity: {best_across_runs.complexity:.6f} | "
            f"Expr: {readable_expr}\n"
        )

print("\nSaved:", ALL_RUNS_BEST_TXT)

In [ ]:
# MULTI RUN -20 RUNS

# ==========================================================
# VALIDATE BEST SINGLE-OBJECTIVE EXPRESSION FROM 20 RUNS
# ==========================================================

import re
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ----------------------------------------------------------
# 1. FILE PATHS
# ----------------------------------------------------------

# result_file = (
#     "/content/drive/My Drive/GWL/r2_complexity/"
#     "train219_val99_1obj_r2 of GP/single_obj_r2_best_per_run.txt"
# )

result_file = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_1obj_r2 of GP/single_obj_best_so_far.txt"
)

validation_path = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_1obj_r2 of GP/validation_test99_dataset.csv"
)

# ----------------------------------------------------------
# 2. LOAD BEST RESULT FOUND SO FAR
# ----------------------------------------------------------

best_r2 = None
best_complexity = None
best_prefix = None

with open(result_file, "r", encoding="utf-8") as f:
    for line in f:

        line = line.strip()

        if line.startswith("R2:"):
            best_r2 = float(line.split(":", 1)[1].strip())

        elif line.startswith("Complexity:"):
            best_complexity = float(
                line.split(":", 1)[1].strip()
            )

        elif line.startswith("Expression:"):
            best_prefix = line.split(":", 1)[1].strip()


if best_r2 is None:
    raise ValueError("Could not find R2 in the result file.")

if best_complexity is None:
    raise ValueError("Could not find complexity in the result file.")

if best_prefix is None:
    raise ValueError("Could not find expression in the result file.")


print("Best result found so far loaded successfully.")
print("Training R2 :", best_r2)
print("Complexity  :", best_complexity)


# ----------------------------------------------------------
# 3. CORRECTED PREFIX PARSER
#    Protects feature names that contain spaces
# ----------------------------------------------------------

def parse_prefix(expr_str):

    # Protect multi-word feature names before tokenization
    replacements = {
        "Lineament density": "Lineament_density",
        "Drainage density": "Drainage_density"
    }

    for old, new in replacements.items():
        expr_str = expr_str.replace(old, new)

    tokens = (
        expr_str
        .replace('(', ' ( ')
        .replace(')', ' ) ')
        .replace(',', ' ')
        .split()
    )

    def parse_tokens(tokens):

        if not tokens:
            return None

        token = tokens.pop(0)

        # Restore original feature names
        if token == "Lineament_density":
            token = "Lineament density"

        elif token == "Drainage_density":
            token = "Drainage density"

        if token in OPERATORS:

            func, arity = OPERATORS[token]

            children = [
                parse_tokens(tokens)
                for _ in range(arity)
            ]

            return Node(token, children)

        elif token not in ['(', ')']:

            return Node(token)

        else:

            return parse_tokens(tokens)

    return parse_tokens(tokens)


# ----------------------------------------------------------
# 4. RECONSTRUCT THE WINNING EXPRESSION TREE
# ----------------------------------------------------------

best_tree = parse_prefix(best_prefix)

print("Expression tree reconstructed successfully.")
print("Tree root:", best_tree.name)

# ----------------------------------------------------------
# 5. LOAD VALIDATION DATA
# ----------------------------------------------------------

FEATURE_COLUMNS = [
    'Soil',
    'Slope',
    'Rain',
    'LULC',
    'Lithology',
    'Lineament density',
    'Geomorphology',
    'Drainage density'
]

TARGET_COLUMN = 'Net Groundwater Recharge'

df_val = pd.read_csv(validation_path)

X_val = df_val[FEATURE_COLUMNS].values
y_val = df_val[TARGET_COLUMN].values

feature_map = {
    name: idx
    for idx, name in enumerate(FEATURE_COLUMNS)
}

print("Validation samples:", len(df_val))

# ----------------------------------------------------------
# 6. EVALUATE THE WINNING EXPRESSION
# ----------------------------------------------------------

preds = []

for row in X_val:

    pred = best_tree.evaluate(
        row,
        feature_map
    )

    preds.append(pred)

preds = np.array(preds, dtype=float)

# Check for invalid values
if np.any(np.isnan(preds)):
    print("Warning: NaN values found in predictions.")

if np.any(np.isinf(preds)):
    print("Warning: Infinite values found in predictions.")

# ----------------------------------------------------------
# 7. CALCULATE VALIDATION METRICS
# ----------------------------------------------------------

validation_r2 = r2_score(
    y_val,
    preds
)

mse_val = mean_squared_error(
    y_val,
    preds
)

rmse_val = np.sqrt(
    mse_val
)

mae_val = mean_absolute_error(
    y_val,
    preds
)

# ----------------------------------------------------------
# 8. PRINT RESULTS
# ----------------------------------------------------------

print("\n===== SINGLE-OBJECTIVE VALIDATION RESULT =====")

print(f"Training R2   : {best_r2:.6f}")
print(f"Validation R2 : {validation_r2:.6f}")
print(f"Complexity    : {best_complexity:.6f}")
print(f"MSE           : {mse_val:.6f}")
print(f"RMSE          : {rmse_val:.6f}")
print(f"MAE           : {mae_val:.6f}")


# ----------------------------------------------------------
# 9. SAVE VALIDATION RESULT TO TXT FILE
# ----------------------------------------------------------

output_txt = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_1obj_r2 of GP/"
    "single_obj_validation_r2.txt"
)

with open(output_txt, "w", encoding="utf-8") as f:

    f.write("SINGLE-OBJECTIVE SYMBOLIC REGRESSION VALIDATION RESULT\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"Training R2   : {best_r2:.6f}\n")
    f.write(f"Validation R2 : {validation_r2:.6f}\n")
    f.write(f"Complexity    : {best_complexity:.6f}\n")
    f.write(f"MSE           : {mse_val:.6f}\n")
    f.write(f"RMSE          : {rmse_val:.6f}\n")
    f.write(f"MAE           : {mae_val:.6f}\n")

    f.write("\nPrefix Expression:\n")
    f.write(best_prefix + "\n")

print("\nValidation result saved to:")
print(output_txt)

In [ ]:
# MULTI RUN -20 RUNS

#==========================================================
# SAVE SINGLE-OBJECTIVE VALIDATION DETAILS TO EXCEL
# ==========================================================

import os
import pandas as pd

excel_path = (
    "/content/drive/My Drive/GWL/r2_complexity/"
    "train219_val99_1obj_r2 of GP/"
    "single_obj_validation_results.xlsx"
)

os.makedirs(os.path.dirname(excel_path), exist_ok=True)

# ----------------------------------------------------------
# Prepare validation data
# ----------------------------------------------------------

df_result = df_val[FEATURE_COLUMNS].copy()

df_result["Actual"] = y_val
df_result["Predicted"] = preds

# ----------------------------------------------------------
# Save to Excel
# ----------------------------------------------------------

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    # Write validation observations starting from row 3
    df_result.to_excel(
        writer,
        sheet_name="Best_Expression",
        index=False,
        startrow=2
    )

    # Access worksheet
    ws = writer.sheets["Best_Expression"]

    # First row: main result information
    ws["A1"] = (
        f"Training R²: {best_r2:.6f}, "
        f"Validation R²: {validation_r2:.6f}, "
        f"Complexity: {best_complexity:.6f}, "
        f"MSE: {mse_val:.6f}, "
        f"RMSE: {rmse_val:.6f}, "
        f"MAE: {mae_val:.6f}, "
        f"Expression: {best_prefix}"
    )

print("\nExcel file saved to:")
print(excel_path)

In [ ]:
# MULTI RUN -20 RUNS

## prefix to infix

import re

#input_file = "/content/drive/MyDrive/GWL/r2_complexity/train219_val99_1obj_r2 of GP/single_obj_r2_best_per_run.txt"
input_file = "/content/drive/MyDrive/GWL/r2_complexity/train219_val99_1obj_r2 of GP/single_obj_best_so_far.txt"
output_txt = "/content/drive/MyDrive/GWL/r2_complexity/train219_val99_1obj_r2 of GP/single_obj_best_infix.txt"

def split_args(s: str):
    args, depth, start = [], 0, 0
    for i, ch in enumerate(s):
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        elif ch == ',' and depth == 0:
            args.append(s[start:i].strip())
            start = i + 1
    args.append(s[start:].strip())
    return args

def prefix_to_infix(expr: str) -> str:
    expr = expr.strip()

    # number
    if re.fullmatch(r"-?\d+(\.\d+)?", expr):
        return expr

    # variable name (underscore ok)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", expr):
        return expr

    # general call name(...)
    m = re.match(r"^([A-Za-z_+\-*/][A-Za-z0-9_]*)\((.*)\)$", expr)
    if not m:
        return expr

    name = m.group(1)
    inside = m.group(2).strip()
    args = split_args(inside)
    args_infix = [prefix_to_infix(a) for a in args]

    # binary ops
    if name in {"+", "-", "*", "/"} and len(args_infix) == 2:
        a, b = args_infix
        return f"({a} {name} {b})"

    # power
    if name == "pow" and len(args_infix) == 2:
        a, b = args_infix
        return f"({a} ^ {b})"

    # unary neg (if your GP uses neg(x))
    if name == "neg" and len(args_infix) == 1:
        return f"(-{args_infix[0]})"

    # other functions: cos(x), square(x), cube(x), exp(x), ...
    return f"{name}({', '.join(args_infix)})"

# -----------------------------
# READ BEST RESULT FOUND SO FAR
# -----------------------------
best_r2 = None
best_complexity = None
best_prefix = None

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:

        line = line.strip()

        if line.startswith("R2:"):
            best_r2 = line.split(":", 1)[1].strip()

        elif line.startswith("Complexity:"):
            best_complexity = line.split(":", 1)[1].strip()

        elif line.startswith("Expression:"):
            best_prefix = line.split(":", 1)[1].strip()


if best_r2 is None:
    raise ValueError("Could not find R2 in the result file.")

if best_complexity is None:
    raise ValueError("Could not find complexity in the result file.")

if best_prefix is None:
    raise ValueError("Could not find expression in the result file.")


best_infix = prefix_to_infix(best_prefix)

# Print to console
print("===== BEST SINGLE-OBJECTIVE RESULT FOUND SO FAR =====")
print("R2        :", best_r2)
print("Complexity:", best_complexity)

print("\nPrefix:\n", best_prefix)
print("\nInfix:\n", best_infix)


# Save to TXT
with open(output_txt, "w", encoding="utf-8") as f:

    f.write("BEST SINGLE-OBJECTIVE RESULT FOUND SO FAR\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"R2: {best_r2}\n")
    f.write(f"Complexity: {best_complexity}\n\n")

    f.write("Prefix Expression:\n")
    f.write(best_prefix + "\n\n")

    f.write("Infix Expression:\n")
    f.write(best_infix + "\n")


print("\nSaved TXT to:", output_txt)

In [ ]:
# ==========================================================
# SAVE SINGLE-OBJECTIVE BEST EXPRESSION AS LATEX TABLE
# Includes Solution column
# ==========================================================

output_tex = (
    "/content/drive/MyDrive/GWL/r2_complexity/"
    "train219_val99_1obj_r2 of GP/"
    "single_obj_best_latex.tex"
)

def infix_to_latex(expr):
    """Convert infix expression to LaTeX format."""

    expr = expr.replace('&', r'\&')
    expr = expr.replace('%', r'\%')
    expr = expr.replace('_', r'\_')

    expr = expr.replace('sin(', r'\sin(')
    expr = expr.replace('cos(', r'\cos(')
    expr = expr.replace('log(', r'\log(')
    expr = expr.replace('exp(', r'\exp(')

    # Keep multiplication as *
    expr = expr.replace('*', r' * ')

    return expr


latex_expr = infix_to_latex(best_infix)

r2_formatted = f"{float(best_r2):.6f}"
complexity_formatted = f"{float(best_complexity):.2f}"

with open(output_tex, "w", encoding="utf-8") as f:

    f.write(r"\begin{table}[h]" + "\n")
    f.write(r"\centering" + "\n")

    f.write(
        r"\caption{Highest-$R^2$ expression obtained from single-objective symbolic regression}"
        + "\n"
    )

    f.write(r"\label{tab:best_single_obj}" + "\n")

    f.write(r"\setlength{\tabcolsep}{4pt}" + "\n")
    f.write(r"\renewcommand{\arraystretch}{1.1}" + "\n")

    # Four columns:
    # Solution | R² | Complexity | Expression
    f.write(
        r"\begin{tabularx}{\textwidth}"
        r"{|c|c|c|>{\raggedright\arraybackslash}X|}"
        + "\n"
    )

    f.write(r"\hline" + "\n")

    f.write(
        r"Solution & R\textsuperscript{2} & Complexity & Expression \\"
        + "\n"
    )

    f.write(r"\hline" + "\n")

    f.write(
        f"1 & "
        f"{r2_formatted} & "
        f"{complexity_formatted} & "
        f"\\raggedright ${latex_expr}$ \\\\"
        + "\n"
    )

    f.write(r"\hline" + "\n")
    f.write(r"\end{tabularx}" + "\n")
    f.write(r"\end{table}" + "\n")

print("Saved:", output_tex)